##### Support Lakebase & App

Provision Lakebase instance, create synced table for support_agent_reports,
create OLTP tables, set up warehouse, deploy the support console app.

In [ ]:
%pip install databricks-sdk psycopg2-binary --upgrade

In [ ]:
import re
import time
import sys

sys.path.append('../utils')
from uc_state import add

CATALOG = dbutils.widgets.get("CATALOG")
SUPPORT_AGENT_ENDPOINT_NAME = dbutils.widgets.get("SUPPORT_AGENT_ENDPOINT_NAME")

SUPPORT_LAKEBASE_INSTANCE_NAME = re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}supportmanager".lower())

#### Lakebase Instance & Database

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import DatabaseInstance
from databricks.sdk.errors import NotFound, ResourceDoesNotExist

w = WorkspaceClient()

instance_name = SUPPORT_LAKEBASE_INSTANCE_NAME

try:
    existing = w.database.get_database_instance(instance_name)
    print(f"Found existing database instance: {existing.name} (state: {existing.state})")
    instance = existing
except (NotFound, ResourceDoesNotExist):
    print(f"Creating new database instance: {instance_name}")
    instance = w.database.create_database_instance(
        DatabaseInstance(
            name=instance_name,
            capacity="CU_1"
        )
    )
    print(f"Created database instance: {instance.name}")
    add(CATALOG, "databaseinstances", {"name": instance.name})

print("Waiting for database instance to be available...")
for _ in range(60):
    state = str(w.database.get_database_instance(instance_name).state)
    if state == "DatabaseInstanceState.AVAILABLE":
        print(f"Instance ready: {state}")
        break
    time.sleep(10)
else:
    raise TimeoutError(f"Database instance {instance_name} did not become available within 10 minutes")

In [ ]:
from databricks.sdk.service.database import DatabaseCatalog

catalog_name = SUPPORT_LAKEBASE_INSTANCE_NAME

try:
    existing_cat = w.catalogs.get(catalog_name)
    print(f"Found existing catalog: {existing_cat.name}")
except Exception:
    catalog = w.database.create_database_catalog(
        DatabaseCatalog(
            name=catalog_name,
            database_instance_name=SUPPORT_LAKEBASE_INSTANCE_NAME,
            database_name="caspers_support",
            create_database_if_not_exists=True
        )
    )
    print(f"Created new database and catalog: {catalog.name}")
    add(CATALOG, "databasecatalogs", catalog)

#### Synced Table

In [ ]:
from databricks.sdk.service.database import SyncedDatabaseTable, SyncedTableSpec, NewPipelineSpec, SyncedTableSchedulingPolicy

synced_table_name = f"{CATALOG}.support.pg_support_agent_reports"

try:
    existing_table = w.tables.get(synced_table_name)
    print(f"Found existing synced table: {existing_table.full_name}")
except Exception:
    synced_table = w.database.create_synced_database_table(
        SyncedDatabaseTable(
            name=synced_table_name,
            database_instance_name=instance_name,
            logical_database_name="caspers_support",
            spec=SyncedTableSpec(
                source_table_full_name=f"{CATALOG}.support.support_agent_reports",
                primary_key_columns=["support_request_id"],
                scheduling_policy=SyncedTableSchedulingPolicy.CONTINUOUS,
                create_database_objects_if_missing=True,
                new_pipeline_spec=NewPipelineSpec(
                    storage_catalog="storage_catalog",
                    storage_schema="storage_schema"
                )
            ),
        )
    )
    print(f"Created synced table: {synced_table.name}")
    add(CATALOG, "pipelines", synced_table.data_synchronization_status)

#### OLTP Tables (Postgres)

In [ ]:
import psycopg2

# Resolve the Postgres endpoint via the new Postgres API
PROJECT = f"projects/{SUPPORT_LAKEBASE_INSTANCE_NAME}"
branches = list(w.postgres.list_branches(PROJECT))
default_branch = next((b for b in branches if getattr(getattr(b, "status", None), "default", False)), branches[0])
branch_name = default_branch.name

endpoints = list(w.postgres.list_endpoints(parent=branch_name))
if not endpoints:
    for _ in range(12):
        time.sleep(5)
        endpoints = list(w.postgres.list_endpoints(parent=branch_name))
        if endpoints:
            break
    if not endpoints:
        raise RuntimeError("No Lakebase endpoint found")

endpoint = next((e for e in endpoints if str(getattr(getattr(e, "status", None), "endpoint_type", "")).endswith("READ_WRITE")), endpoints[0])
endpoint_name = endpoint.name
endpoint_host = endpoint.status.hosts.host

print(f"Branch: {branch_name}")
print(f"Endpoint: {endpoint_name}")
print(f"Host: {endpoint_host}")

# Connect and create OLTP tables
creds = w.postgres.generate_database_credential(endpoint=endpoint_name)
current_user = w.current_user.me().user_name

conn = psycopg2.connect(
    host=endpoint_host,
    port=5432,
    dbname="databricks_postgres",
    user=current_user,
    password=creds.token,
    sslmode="require",
)
conn.autocommit = False

with conn.cursor() as cur:
    cur.execute("CREATE SCHEMA IF NOT EXISTS support")

    cur.execute("""
        CREATE TABLE IF NOT EXISTS support.operator_actions (
          action_id BIGSERIAL PRIMARY KEY,
          support_request_id TEXT NOT NULL,
          order_id TEXT NOT NULL,
          user_id TEXT,
          action_type TEXT NOT NULL CHECK (action_type IN ('apply_refund','apply_credit','send_reply')),
          amount_usd NUMERIC(10,2),
          payload JSONB,
          status TEXT NOT NULL DEFAULT 'recorded',
          actor TEXT,
          created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS support.support_replies (
          reply_id BIGSERIAL PRIMARY KEY,
          support_request_id TEXT NOT NULL,
          order_id TEXT NOT NULL,
          user_id TEXT,
          message_text TEXT NOT NULL,
          sent_by TEXT,
          created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS support.request_status (
          support_request_id TEXT PRIMARY KEY,
          status TEXT NOT NULL DEFAULT 'pending',
          assigned_to TEXT,
          updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
          last_action TEXT,
          notes TEXT
        )
    """)

    cur.execute("""
        CREATE TABLE IF NOT EXISTS support.response_ratings (
          rating_id BIGSERIAL PRIMARY KEY,
          support_request_id TEXT NOT NULL,
          order_id TEXT NOT NULL,
          user_id TEXT,
          rating TEXT NOT NULL CHECK (rating IN ('thumbs_up','thumbs_down')),
          reason_code TEXT,
          feedback_notes TEXT,
          actor TEXT,
          created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
          UNIQUE (support_request_id)
        )
    """)

    conn.commit()

conn.close()
print("OLTP tables created")

#### Warehouse

In [ ]:
WAREHOUSE_NAME = f"{CATALOG}-warehouse"
existing_wh = [wh for wh in w.warehouses.list() if wh.name == WAREHOUSE_NAME]
if existing_wh:
    warehouse = existing_wh[0]
    print(f"Using existing warehouse: {warehouse.id}")
else:
    warehouse = w.warehouses.create(
        name=WAREHOUSE_NAME,
        cluster_size="2X-Small",
        max_num_clusters=1,
        min_num_clusters=1,
        enable_serverless_compute=True,
    ).result()
    add(CATALOG, "warehouses", warehouse)
    print(f"Created warehouse: {warehouse.id}")

#### App Deployment

In [ ]:
import os

# Write app.yaml with computed values
app_yaml_contents = f"""command: ['npm', 'run', 'start']
env:
  - name: PGPORT
    value: '5432'
  - name: PGDATABASE
    value: 'databricks_postgres'
  - name: PGSSLMODE
    value: 'require'
  - name: LAKEBASE_ENDPOINT
    value: '{endpoint_name}'
  - name: DATABRICKS_WAREHOUSE_ID
    value: '{warehouse.id}'
  - name: SUPPORT_AGENT_ENDPOINT_NAME
    value: '{SUPPORT_AGENT_ENDPOINT_NAME}'
"""

app_yaml_path = os.path.abspath("../apps/supportconsolek/supportconsolek/app.yaml")
if os.path.exists(app_yaml_path):
    os.remove(app_yaml_path)

time.sleep(3)

with open(app_yaml_path, "w") as f:
    f.write(app_yaml_contents)

print(f"Wrote app.yaml with warehouse={warehouse.id}, endpoint={endpoint_name}")

In [ ]:
from databricks.sdk.service.apps import App, AppResource, AppResourceSqlWarehouse, AppResourceDatabase
from databricks.sdk.service.apps import AppResourceSqlWarehouseSqlWarehousePermission, AppResourceDatabaseDatabasePermission

source_code_path = os.path.abspath("../apps/supportconsolek/supportconsolek")
APP_NAME = "supportconsole"

app_def = App(
    name=APP_NAME,
    default_source_code_path=source_code_path,
    resources=[
        AppResource(
            name="sql-warehouse",
            sql_warehouse=AppResourceSqlWarehouse(
                id=warehouse.id,
                permission=AppResourceSqlWarehouseSqlWarehousePermission.CAN_USE),
        ),
        AppResource(
            name="database",
            database=AppResourceDatabase(
                instance_name=SUPPORT_LAKEBASE_INSTANCE_NAME,
                database_name="caspers_support",
                permission=AppResourceDatabaseDatabasePermission.CAN_CONNECT_AND_CREATE,
            ),
        ),
    ],
)

try:
    existing_app = w.apps.get(APP_NAME)
    print(f"App {APP_NAME} already exists, updating...")
    app = w.apps.update(APP_NAME, app_def)
except Exception:
    app = w.apps.create(app_def)

app_status = app.result()
add(CATALOG, "apps", app_status)
print(f"App {APP_NAME} ready")

#### Grants & Postgres Role

In [ ]:
from databricks.sdk.service import catalog as cat

# Grant app permissions to UC catalog/schema/table
w.grants.update(
    full_name=CATALOG,
    securable_type="CATALOG",
    changes=[
        cat.PermissionsChange(
            add=[cat.Privilege.USE_CATALOG],
            principal=app_status.id
        )
    ]
)

w.grants.update(
    full_name=f"{CATALOG}.support",
    securable_type="SCHEMA",
    changes=[
        cat.PermissionsChange(
            add=[cat.Privilege.USE_SCHEMA],
            principal=app_status.id
        )
    ]
)

w.grants.update(
    full_name=f"{CATALOG}.support.support_agent_reports",
    securable_type="TABLE",
    changes=[
        cat.PermissionsChange(
            add=[cat.Privilege.SELECT],
            principal=app_status.id
        )
    ]
)

print("UC grants applied")

In [ ]:
from databricks.sdk.common.types.fieldmask import FieldMask
from databricks.sdk.service.postgres import (
    RoleRoleSpec,
    RoleMembershipRole,
)

# Grant the app's Postgres role DATABRICKS_SUPERUSER
PRODUCTION = list(w.postgres.list_branches(PROJECT))[0].name
APP_ROLE = list(w.postgres.list_roles(PRODUCTION))[1]

APP_ROLE.spec = RoleRoleSpec(
    membership_roles=[RoleMembershipRole.DATABRICKS_SUPERUSER]
)

w.postgres.update_role(
    name=APP_ROLE.name,
    role=APP_ROLE,
    update_mask=FieldMask(["spec.membership_roles"]),
)

print("Postgres role configured")

#### Deploy

In [ ]:
from databricks.sdk.service.apps import AppDeployment

deployment = w.apps.deploy(
    app_name=app_status.name,
    app_deployment=AppDeployment(
        source_code_path=source_code_path
    )
)
deployment_status = deployment.result()
print(f"App deployed: {deployment_status}")